1. Khởi tạo Spark Session

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("MovieRecommendation-ETL-Notebook") \
    .master("local[*]") \
    .getOrCreate()
    
print("=> Khởi tạo Spark thành công!")


=> Khởi tạo Spark thành công!


2. Nạp và làm sạch dữ liệu Rating (u.data)

In [3]:
rating_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("movie_id", IntegerType(), True),
    StructField("rating", IntegerType(), True),
    StructField("timestamp", IntegerType(), True)
])

# Đường dẫn lùi ra 2 thư mục để vào data
rating_path = "../../data/ml-100k/u.data"
ratings_df = spark.read.csv(rating_path, sep="\t", schema=rating_schema)

# Xóa dữ liệu rỗng và trùng lặp
ratings_df = ratings_df.dropna().dropDuplicates()

print(f"Tổng số lượt đánh giá sau làm sạch: {ratings_df.count()}")
ratings_df.show(5) # In thử 5 dòng ra xem


Tổng số lượt đánh giá sau làm sạch: 100000
+-------+--------+------+---------+
|user_id|movie_id|rating|timestamp|
+-------+--------+------+---------+
|    292|     515|     4|881103977|
|    178|     248|     4|882823954|
|     32|     294|     3|883709863|
|     57|     419|     3|883698454|
|    229|     328|     1|891632142|
+-------+--------+------+---------+
only showing top 5 rows



3. Nạp và làm sạch dữ liệu Phim (u.item)

In [4]:
item_path = "../../data/ml-100k/u.item"

# Đã sửa lỗi Encoding thành ISO-8859-1 chuẩn của Spark 3.5
items_df = spark.read.csv(item_path, sep="|", encoding="ISO-8859-1")

# Chỉ trích xuất 2 cột quan trọng nhất: Mã phim và Tên phim
items_df = items_df.select(
    col("_c0").cast(IntegerType()).alias("movie_id"),
    col("_c1").cast(StringType()).alias("movie_title")
)

items_df = items_df.dropna().dropDuplicates(["movie_id"])

print(f"Tổng số phim sau làm sạch: {items_df.count()}")
items_df.show(5, truncate=False)


Tổng số phim sau làm sạch: 1682
+--------+-----------------+
|movie_id|movie_title      |
+--------+-----------------+
|1       |Toy Story (1995) |
|2       |GoldenEye (1995) |
|3       |Four Rooms (1995)|
|4       |Get Shorty (1995)|
|5       |Copycat (1995)   |
+--------+-----------------+
only showing top 5 rows



4. Xuất ra định dạng Parquet siêu tốc

In [5]:
output_rating_path = "../../data/processed_ratings.parquet"
output_item_path = "../../data/processed_items.parquet"

# Ghi đè file Parquet
ratings_df.write.mode("overwrite").parquet(output_rating_path)
items_df.write.mode("overwrite").parquet(output_item_path)

print("=> Tuyệt vời! Đã xuất thành công 2 file Parquet để dành cho EDA và Machine Learning.")


=> Tuyệt vời! Đã xuất thành công 2 file Parquet để dành cho EDA và Machine Learning.
